# Module 4 - Session 1 -- Lab Notebook
## API Usage, Request/Response Handling & Building Scripts -- "The Client Library Workshop"

**Course:** Generative & Agentic AI Systems
**Time:** ~60 minutes (in-session) - seeds this session's assignment
**Runs on:** Google Colab, Kaggle, or your own laptop

---

## 🏢 Welcome to Northstar AI

It's your first week as a backend engineer at **Northstar AI**, a small startup bolting AI features onto a product that used to be a plain old CRUD app. There's no ML platform team yet -- just you, a handful of other engineers, and a Slack channel called `#prod-fires`.

Your manager, **Priya**, drops by your desk with your onboarding ticket:

> "Welcome aboard. First thing: we call an LLM API directly from a few places in the codebase, and it's already bitten us once -- a teammate's script hammered the provider, got rate-limited, and the whole thing just... died. No retry, no logging, nobody noticed until a customer complained. I want you to build the client wrapper we should have had from day one. Today's session gets you through the fundamentals; the assignment has you harden it further."

**What you are actually building today, in one sentence:** a single class, `LLMClient`, that every part of Northstar AI's codebase will import instead of calling the `openai` SDK directly. You write it in two required pieces (Ticket #2's `chat()` and Ticket #4's `process_batch()`); everything else in the notebook either prepares you to write those pieces correctly or proves they work.

**The big idea of today:** so far you've called a model running on the same machine as your code (GPT-2/BERT, locally, via Hugging Face). Today you learn to call a model running on **someone else's server**, over the internet -- which means dealing with authentication, real network errors, and real rate limits for the first time. Exactly the class of bug that bit Northstar AI last month.

**You will need a free Groq API key for this lab.** Every call in this notebook is real -- there is no simulator or mock server. This is intentional: real APIs behave in ways a scripted fake never quite matches (timing, error bodies, rate-limit quirks), and that's exactly what you're here to learn to handle. Priya isn't going to accept "it worked in the simulator" as proof it's production-ready, and neither should you.

### Getting your free key (under a minute)
1. Go to [console.groq.com](https://console.groq.com) and sign up -- no credit card required
2. Create an API key from the console
3. Set it as an environment variable or Colab secret named `GROQ_API_KEY` -- **never paste it directly into a code cell** (Rule Zero, below)

**Words we'll use a lot today:**

| Term | Plain-language meaning |
|---|---|
| API | A defined way for your code to ask another program (often on a server) to do something and hand back a result |
| Endpoint | The specific URL/address you send a request to |
| API key | A secret string proving a request is really from your account |
| Rate limit | A cap on how many requests you can send per minute |
| Retry with backoff | Trying again after a failure, waiting a little longer each time |
| OpenAI-compatible API | A provider whose API uses the same request/response shape as OpenAI's -- Groq is one, so the `openai` Python package works against it directly |

### Your onboarding tickets today
| Ticket | What you build | Time | Priority |
|---|---|---|---|
| #1 | A small helper, `summarize_usage()`, that audits token spend across calls | 12 min | 🟢 core |
| #2 | **`LLMClient.chat()`** -- the core method: retry-with-backoff, fail-fast on client errors, AND a hard cost cap, all three in one method you write | 20 min | 🟢 core |
| #3 | *(no new code)* -- you PROVE #2 survives three real failure types (401, 400, 429) | 12 min | 🟢 core |
| #4 | **`process_batch()`** -- checkpointing, structured logging, and a circuit breaker that stops a batch after repeated failures, all in one method you write | 16 min | 🟢 core |

> **A note on what "verified" means in this notebook:** every cell here is written to be correct against Groq's documented, OpenAI-compatible API. If a specific error code or rate-limit number doesn't match exactly what you see (providers adjust limits over time), that's expected -- focus on the PATTERN (how your code responds to each error class), not the exact numbers. Priya cares about the same thing: she'll ask "what does this do when it fails?", not "what was today's exact error message?" Every build cell below has an **acceptance test** (asserts) right under the TODOs -- don't edit those, they're the proof your code actually does the job, not just that it runs.

In [2]:
import os
import time
import logging

from openai import OpenAI, AuthenticationError, RateLimitError, BadRequestError, APIStatusError
from google.colab import userdata

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except userdata.SecretNotFoundError:
        # If secret not found, GROQ_API_KEY remains None or empty string
        pass

if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. Set it as a Colab secret or environment variable before continuing -- "
        "see the intro above for the 1-minute signup steps at console.groq.com."
    )

MODEL = "openai/gpt-oss-20b"  # small, fast, generous free-tier limits -- good for this lab
BASE_URL = "https://api.groq.com/openai/v1"

print("Setup complete -- GROQ_API_KEY found, ready to make real calls.")

Setup complete -- GROQ_API_KEY found, ready to make real calls.


---
## 🎫 Ticket #1 -- "Show me you understand what we're actually sending" (10 min) 🟢 core

Priya's rule for new hires: nobody touches the client wrapper until they've made one raw call and read every field of the response themselves. "I've debugged too many bugs where someone assumed a field existed," she says. "Look at the real thing first."

Let's make one real call and look at exactly what comes back -- this is the payload structure every part of this lab builds on.

**📝 Notes -- what's actually in an API response**
- An LLM API call is just an HTTP request/response, the same shape as any REST API you've used before. The `openai` SDK just wraps the JSON in Python objects for convenience -- `response.choices[0].message.content` is really `response_json["choices"][0]["message"]["content"]`.
- `usage` isn't decoration -- it's literally what you're billed on. `prompt_tokens` tracks input length (including any system prompt / conversation history), `completion_tokens` tracks output length, and providers usually charge more per output token than per input token.
- `finish_reason` tells you *why* generation stopped: `"stop"` = the model finished naturally, `"length"` = it got cut off by your `max_tokens` limit. (That's why the call above can return a near-empty-looking reply -- `max_tokens=20` wasn't enough room for this model's internal reasoning tokens before it had to stop.)

In [3]:
client = OpenAI(api_key=GROQ_API_KEY, base_url=BASE_URL)

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of Japan? Answer in one word."}],
    temperature=0.3,
    max_tokens=20,
)

print("Reply:", response.choices[0].message.content)
print("\nFull response object (the important fields):")
print(" id:", response.id)
print(" finish_reason:", response.choices[0].finish_reason)
print(" usage:", response.usage)

Reply: 

Full response object (the important fields):
 id: chatcmpl-e9f2a7c8-b53f-42a7-9f23-edc4bdd360d7
 finish_reason: length
 usage: CompletionUsage(completion_tokens=20, prompt_tokens=83, total_tokens=103, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=18, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.022073215, prompt_time=0.003949723, completion_time=0.021690135, total_time=0.025639858)


> **Checkpoint 1:** Compare the printed fields to Slide 8's worked example -- same shape: `choices[0].message.content` for the reply, `usage` for token counts. This is a REAL response from Groq's real servers, not a simulated one.

**🔧 Troubleshooting:**
- `AuthenticationError` here means your `GROQ_API_KEY` is missing or invalid -- double check your Colab secret / environment variable name matches exactly.
- If the cell hangs, check your internet connection; there's no local model to fall back on today.

**⚠️ Note on randomness:** the exact wording of `response.choices[0].message.content` may vary slightly between runs even at low temperature (Module 2 Session 2's sampling material) -- that's real model behavior, not a bug.

**💬 Priya, over Slack:** "One more thing -- finance will ask about token spend eventually, and they'll ask *you* first since you own the client. `usage.completion_tokens` is basically what you're paying for on the output side. Get comfortable reading it now, before it's your problem during an incident."

**✏️ Exercise 1.1 -- Build: a usage auditor**

Priya's actual ask: "Before finance asks, I want ONE function that tells me whether a batch of calls blew the token budget -- not me eyeballing `usage` fields one call at a time."

Build `summarize_usage(responses, token_budget)` that:
1. Takes a **list** of raw response objects (like the one from Ticket #1) and an integer `token_budget`.
2. Returns a dict with `total_prompt_tokens`, `total_completion_tokens`, `total_tokens`, and `over_budget` (`True`/`False`).
3. Does **not** make any API calls itself -- it only reads `.usage` off responses it's given.

Then make 3 real calls with different prompts, pass the three responses into your function, and prove it with asserts (below -- don't edit those, they're your acceptance test).

In [4]:
# ✏️ Exercise 1.1 -- your code here

def summarize_usage(responses, token_budget):
    total_prompt_tokens = sum(r.usage.prompt_tokens for r in responses)
    total_completion_tokens = sum(r.usage.completion_tokens for r in responses)
    total_tokens = sum(r.usage.total_tokens for r in responses)

    return {
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "over_budget": total_tokens > token_budget
    }
    # TODO: replace this stub
    raise NotImplementedError

# Make 3 real calls with three DIFFERENT prompts of your choosing.
prompts_1_1 = [
    "Captial of cario ",
    "Captial of spain",
    "Captial of greek"
]
responses_1_1 = [
    client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": p}], max_tokens=60)
    for p in prompts_1_1
]

summary = summarize_usage(responses_1_1, token_budget=100)
print(summary)

# --- Acceptance test: do not edit below this line ---
expected_total = sum(r.usage.total_tokens for r in responses_1_1)
expected_prompt = sum(r.usage.prompt_tokens for r in responses_1_1)
expected_completion = sum(r.usage.completion_tokens for r in responses_1_1)

assert summary["total_prompt_tokens"] == expected_prompt, "prompt token sum doesn't match"
assert summary["total_completion_tokens"] == expected_completion, "completion token sum doesn't match"
assert summary["total_tokens"] == expected_total, "total token sum doesn't match"
assert summary["over_budget"] == (expected_total > 100), "over_budget flag is wrong"

# A budget your calls definitely blow, to prove the flag actually flips:
tiny_budget_summary = summarize_usage(responses_1_1, token_budget=1)
assert tiny_budget_summary["over_budget"] is True, "over_budget should be True when budget=1"

print("✅ PASSED -- summarize_usage() correctly totals usage and flags budget overruns.")

{'total_prompt_tokens': 228, 'total_completion_tokens': 166, 'total_tokens': 394, 'over_budget': True}
✅ PASSED -- summarize_usage() correctly totals usage and flags budget overruns.


---
## 🎫 Ticket #2 -- "One client class, not five copy-pasted call sites" (15 min) 🟢 core

Priya pulls up a code search: `client.chat.completions.create(` shows up in four different files, each with slightly different (or missing) error handling. That's exactly the mess that caused last month's outage -- nobody had one place to fix it.

> "Build us the class from Slide 16. Config in one place, retry logic in one place. When we need to change how we handle failures, I want to change it once, not hunt through the codebase."

Time to build it. We'll do it in small, understandable pieces, then assemble them.

### Piece 1: basic structure (config lives in `__init__`)

**📝 Notes -- why "one class," and why retries need rules**
- **Why one class, not five call sites:** every place that calls the API needs to agree on the same error-handling policy. Duplicate it five times and you get five slightly different bugs. Centralizing it means Priya's "change it once" requirement is actually possible.
- **Why NOT retry everything:** an HTTP status code tells you *whose fault* the failure is.
  - **4xx (401, 400)** = client error -- the request itself was wrong. No amount of retrying fixes a bad password or an invalid parameter; you're just wasting time and hiding a bug that needs a human to fix.
  - **429 / 5xx** = the server is temporarily rate-limiting or overloaded -- exactly the kind of failure that often resolves itself if you wait and try again.
- **Why exponential backoff, not a fixed delay:** if everyone hitting a rate limit retried after exactly 1 second, they'd all retry at the same moment and re-trigger the limit -- a "thundering herd." Growing the wait each attempt (1s, 2s, 4s...) spreads retries out and gives the server room to recover.
- **Why the cost cap lives inside `chat()`, not somewhere else:** a safety rule that callers can forget to apply themselves isn't really a safety rule. Putting it inside the one method everyone is forced to call through means it can't be skipped.

In [5]:
class LLMClient:
    def __init__(self, api_key, model=MODEL, max_retries=3, base_delay=1.0, max_tokens_cap=500):
        """
        api_key: your Groq API key
        model: which model to request
        max_retries: how many times to retry a failed call before giving up
        base_delay: starting wait time (seconds) for exponential backoff
        max_tokens_cap: hard ceiling on max_tokens a caller can request (Priya's cost-safety rule --
                        you'll enforce this in chat() below)
        """
        self.client = OpenAI(api_key=api_key, base_url=BASE_URL)
        self.model = model
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_tokens_cap = max_tokens_cap
        self.call_log = []  # you'll build this up inside chat(), one entry per attempt

    def chat(self, messages, **kwargs):
        """The main method every caller uses. You build this in the next cell."""
        raise NotImplementedError("Build retry + fail-fast + cap logic in the next cell!")

print("LLMClient skeleton defined -- chat() is still a stub, you'll build it next.")

LLMClient skeleton defined -- chat() is still a stub, you'll build it next.


### Piece 2 -- ✏️ YOU build `chat()`

This is the method Priya actually cares about most, and the one every other cell in this notebook tests or depends on. Build `chat(self, messages, **kwargs)` on `LLMClient` that does **all three** of the following:

**1. Enforce the cost cap FIRST, before any network call.**
If `kwargs.get("max_tokens")` is set and exceeds `self.max_tokens_cap`, raise `ValueError` immediately. Nothing gets sent to Groq.

**2. Retry with exponential backoff (Slide 14): `wait = base_delay * 2**attempt`.**
Loop up to `self.max_retries` times, calling `self.client.chat.completions.create(model=self.model, messages=messages, **kwargs)`:
- **On success:** append `{"status": "success", "attempt": attempt + 1}` to `self.call_log` and return the response.
- **On `AuthenticationError` or `BadRequestError`** (401/400 -- OUR code's fault; retrying won't fix a wrong key or an invalid parameter): append `{"status": "failed_fast", "error": str(e)}` and re-raise immediately -- no sleep, no more attempts. **This except clause must come before the one below** -- both of these exception types are technically a *kind* of `APIStatusError` too, and Python tries `except` clauses in order, so the specific case has to be checked first or it never runs.
- **On `RateLimitError` or `APIStatusError`** (429, or 500/503 -- worth retrying): append `{"status": "retry", "attempt": attempt + 1, "error": str(e)}`, `time.sleep(wait)`, then loop again.
- **If the loop runs out of attempts:** append `{"status": "exhausted", "error": str(last_error)}` and re-raise the last error.

**Your `call_log` schema is load-bearing** -- Ticket #3's tests and Exercise 3.1 later in this notebook assert against these exact status strings (`"success"`, `"failed_fast"`, `"retry"`, `"exhausted"`) and the `"attempt"` key, so match them precisely.

In [6]:
# ✏️ Piece 2 -- your code here

def chat(self, messages, **kwargs):
    # TODO 1:
    if kwargs.get("max_tokens") is not None:
        if kwargs["max_tokens"] > self.max_tokens_cap:
            raise ValueError(
                f"max_tokens exceeds the cap of {self.max_tokens_cap}"
            )

    last_error = None

    # TODO 2: retry loop with exponential backoff. See the spec above for the exact

    for attempt in range(self.max_retries):

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                **kwargs
            )

            self.call_log.append({ "status": "success","attempt": attempt + 1})
            return response

        except (AuthenticationError, BadRequestError) as e:
            self.call_log.append({
                "status": "failed_fast",
                "error": str(e)
            })
            raise  # terminate
        except (RateLimitError, APIStatusError) as e:

            last_error = e
            self.call_log.append({
                "status": "retry",
                "attempt": attempt + 1,
                "error": str(e)
            })
            wait = self.base_delay * (2 ** attempt)
            time.sleep(wait)

    self.call_log.append({
        "status": "exhausted",
        "error": str(last_error)
    })
    raise last_error

# Attach this method to the class we defined above.
LLMClient.chat = chat
print("chat() defined -- run the acceptance test in the next cell to check your work.")

chat() defined -- run the acceptance test in the next cell to check your work.


### 🔍 Acceptance test -- does `chat()` actually do the job?

This checks the two things every caller will rely on: a normal call still works, and the cap really is checked before the network call. Ticket #3 (next) does the deeper, real-failure testing -- this is just the smoke test.

In [7]:
# --- Acceptance test: do not edit ---

my_client = LLMClient(api_key=GROQ_API_KEY)
response = my_client.chat(messages=[{"role": "user", "content": "Hello!"}])
print("Reply:", response.choices[0].message.content)
print("Call log:", my_client.call_log)

assert response.choices[0].message.content, "expected a non-empty reply"
assert my_client.call_log == [{"status": "success", "attempt": 1}], \
    f"call_log entry doesn't match the required schema: {my_client.call_log}"
print("✅ Normal call succeeded with the correct call_log schema.")

# Cap must be enforced BEFORE any network call. Prove it with a bad key + a huge max_tokens:
# if the cap check runs first, you get ValueError. If it doesn't, the bad key hits the
# network first and you'd get AuthenticationError instead -- proof the check ran too late.
cap_client = LLMClient(api_key="invalid-key-definitely-wrong-12345", base_delay=0.3)
try:
    cap_client.chat(messages=[{"role": "user", "content": "test"}], max_tokens=99999)
    raise AssertionError("Expected a ValueError -- no exception was raised at all.")
except ValueError:
    print("✅ PASSED -- max_tokens_cap was enforced locally, with zero network calls.")
except AuthenticationError:
    raise AssertionError(
        "Got AuthenticationError instead of ValueError -- your cap check is running "
        "AFTER the network call, not before it. Move it to the top of chat()."
    )

Reply: Hello! 👋 How can I help you today?
Call log: [{'status': 'success', 'attempt': 1}]
✅ Normal call succeeded with the correct call_log schema.
✅ PASSED -- max_tokens_cap was enforced locally, with zero network calls.


> **Checkpoint 2:** both asserts above should have passed. If they didn't, fix `chat()` before moving on -- Ticket #3 builds directly on it, throwing real failures at the exact method you just wrote. This is your real client, making real calls, through code you wrote.

---
## 🎫 Ticket #3 -- "Prove it, don't tell me" (10 min) 🟢 core

Priya won't approve a PR on "trust me, the retry logic works." Her actual line: "Show me it surviving a real failure, not a mocked one -- mocks are exactly how the last version passed review and still broke."

So: we're going to trigger THREE real error types on purpose, using the real Groq API, and watch your client handle each one correctly. This is the evidence that goes in the pull request description.

**📝 Notes -- why "prove it" means real failures, not mocks**
- A mock only tests what you *assumed* would happen. Real APIs have real quirks -- exact error bodies, timing, sometimes a differently-shaped error than the docs promised. Code that only survives a mock can still break in production the first moment reality disagrees with your assumption.
- The three tests below map directly to the two branches you wrote in `chat()`: Tests A and B prove the fail-fast branch triggers on genuinely-unfixable errors; Test C proves the retry branch triggers *and recovers* from a genuinely transient one.
- Notice Test A doesn't need your real account -- a wrong key is wrong everywhere, so fail-fast behavior is fully reproducible. Test C DOES need real account traffic, because a rate limit is a property of your specific account's usage in that moment, not something you can fake locally.

### Test A -- a real 401 (bad key, should fail fast, NOT retry)

**Scenario:** imagine a deploy pushes a rotated key before the secret manager finishes updating, and for a few seconds every request uses a stale key. Priya doesn't want the client to burn three retries and 15 seconds of backoff on something that will *never* succeed -- she wants it to fail loud and immediate so the on-call engineer sees it right away.

We deliberately use a wrong key string. This doesn't need your real account -- Groq's server genuinely rejects it, immediately.

In [8]:
bad_key_client = LLMClient(api_key="invalid-key-definitely-wrong-12345", base_delay=0.3)

try:
    bad_key_client.chat(messages=[{"role": "user", "content": "test"}])
except AuthenticationError as e:
    print(f"Correctly raised immediately (real 401): {e}")

print("Log:", bad_key_client.call_log, " <- should show ONE entry, not three")
assert len(bad_key_client.call_log) == 1 and bad_key_client.call_log[0]["status"] == "failed_fast"
print("✅ PASSED -- a real bad key failed fast, with zero wasted retries.")

Correctly raised immediately (real 401): Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
Log: [{'status': 'failed_fast', 'error': "Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}"}]  <- should show ONE entry, not three
✅ PASSED -- a real bad key failed fast, with zero wasted retries.


### Test B -- a real 400 (invalid parameter, should also fail fast)

**Scenario:** a frontend engineer wires up a "creativity slider" in the product UI and forgets to clamp it -- it sends `temperature=5.0` straight through. `temperature` must be between 0 and 2 for OpenAI-compatible APIs, so the request is simply invalid. We deliberately send that same invalid value.

In [9]:
bad_request_client = LLMClient(api_key=GROQ_API_KEY, base_delay=0.3)

try:
    bad_request_client.chat(messages=[{"role": "user", "content": "test"}], temperature=5.0)
except BadRequestError as e:
    print(f"Correctly raised immediately (real 400): {e}")
except Exception as e:
    # Some providers validate this slightly differently -- if you see a different
    # exception type here, print it and compare to the error-code table on Slide 14.
    print(f"Got a different exception -- inspect it: {type(e).__name__}: {e}")

print("Log:", bad_request_client.call_log)

Correctly raised immediately (real 400): Error code: 400 - {'error': {'message': "'temperature' : number must be at most 2", 'type': 'invalid_request_error'}}
Log: [{'status': 'failed_fast', 'error': 'Error code: 400 - {\'error\': {\'message\': "\'temperature\' : number must be at most 2", \'type\': \'invalid_request_error\'}}'}]


### Test C -- a real 429 (rate limit, SHOULD retry and recover)

**Scenario:** this is the actual incident that got you this ticket -- a script fired off requests back-to-back with zero delay and got rate-limited into oblivion. Let's reproduce it on purpose and confirm your client survives it this time.

This is the one that genuinely can't be scripted deterministically -- we call faster than the free-tier limit allows and see what actually happens. **Your exact numbers will differ from your neighbor's** -- that's expected and realistic, not a bug (Slide 22's field guide).

In [10]:
# Fire off rapid real calls with essentially no delay between them.
# Depending on your account's current free-tier RPM for this model, you should see
# at least one real 429 partway through -- watch for retry/backoff messages below.
rapid_client = LLMClient(api_key=GROQ_API_KEY, base_delay=0.5, max_retries=4)

for i in range(15):
    try:
        r = rapid_client.chat(messages=[{"role": "user", "content": f"Say the number {i}."}], max_tokens=5)
        print(f"Call {i}: OK -- {r.choices[0].message.content.strip()}")
    except RateLimitError:
        print(f"Call {i}: exhausted retries on a real 429 -- this can happen if you're right at your limit")

retry_events = [e for e in rapid_client.call_log if e["status"] == "retry"]
print(f"\nTotal real retry events triggered: {len(retry_events)}")
if retry_events:
    print("✅ You triggered at least one REAL rate limit and your client recovered from it.")
else:
    print("No rate limit hit this run -- your account's current limit is higher than 15 rapid calls. Try increasing the range above, or run it again immediately.")

Call 0: OK -- 
Call 1: OK -- 
Call 2: OK -- 
Call 3: OK -- 
Call 4: OK -- 
Call 5: OK -- 
Call 6: OK -- 
Call 7: OK -- 
Call 8: OK -- 
Call 9: OK -- 
Call 10: OK -- 
Call 11: OK -- 
Call 12: OK -- 
Call 13: OK -- 
Call 14: OK -- 

Total real retry events triggered: 0
No rate limit hit this run -- your account's current limit is higher than 15 rapid calls. Try increasing the range above, or run it again immediately.


**🔧 Troubleshooting:** if Test C never triggers a 429 no matter how many calls you send, your account may be on a higher-limit tier, or Groq's current limits for this model are generous. That's fine -- the important thing is that your code is CORRECT for when it does happen (proven by Tests A and B's fail-fast behavior using the same exception-handling branches).

**✏️ Exercise 3.1 -- Build: a backoff-time calculator**

Priya's next question after seeing Test C: "Fine, it retries. How much wall-clock time did that actually burn? If we're paying for a worker to sit idle during backoff, I want that number, not a vibe."

Build `total_wait_time(call_log, base_delay)`:
1. Reads a `call_log` (the same list your `LLMClient` produces) and a `base_delay`.
2. Sums the actual backoff wait for every `"retry"` entry, using the exact formula from `chat_with_retry` above: `wait = base_delay * (2 ** attempt)`, where `attempt` is 0-indexed but the log stores `attempt + 1` -- so you need `base_delay * (2 ** (entry["attempt"] - 1))`.
3. Ignores `"success"`, `"failed_fast"`, and `"exhausted"` entries -- they don't have a wait attached to them in this calculation.

Since real rate limits are non-deterministic (as the note above says), first prove your function against a **fixed, hand-written log** where you know the right answer -- then optionally run it against `rapid_client.call_log` for real, informational numbers.

In [11]:
# ✏️ Exercise 3.1 -- your code here

def total_wait_time(call_log, base_delay):
    total = 0

    for entry in call_log:
        if entry["status"] == "retry":
            attempt = entry["attempt"]
            wait = base_delay * (2 ** (attempt - 1))
            total += wait
    return total

# --- Acceptance test: do not edit below this line ---

# A hand-written log: two retries (attempt 1, attempt 2) then a success (attempt 3).
fixed_log = [
    {"status": "retry", "attempt": 1, "error": "429"},
    {"status": "retry", "attempt": 2, "error": "429"},
    {"status": "success", "attempt": 3},
]
# attempt 1 -> wait = base_delay * 2**0 = base_delay
# attempt 2 -> wait = base_delay * 2**1 = 2 * base_delay
# total = 3 * base_delay
assert total_wait_time(fixed_log, base_delay=1.0) == 3.0, "wrong total for base_delay=1.0"
assert total_wait_time(fixed_log, base_delay=0.5) == 1.5, "wrong total for base_delay=0.5"

# A log with no retries at all (e.g. Test A's fail-fast case) should cost zero wait time.
assert total_wait_time(bad_key_client.call_log, base_delay=0.3) == 0.0, \
    "a fail-fast log should have zero backoff time"

print("✅ PASSED -- total_wait_time() matches the exponential-backoff formula.")

# Now apply it to your real Test C run, for real numbers (informational -- not asserted,
# since whether you hit a 429 at all depends on your account's current limits):
real_wait = total_wait_time(rapid_client.call_log, base_delay=0.5)
print(f"Real backoff time burned in Test C: {real_wait:.1f}s "
      f"(from {len([e for e in rapid_client.call_log if e['status']=='retry'])} real retries)")

✅ PASSED -- total_wait_time() matches the exponential-backoff formula.
Real backoff time burned in Test C: 0.0s (from 0 real retries)


**📝 Notes -- why these three features belong together**
- **Checkpointing** exists for one reason: a batch job that dies at item 150 of 200 without checkpointing has to redo all 150 -- burning time and money re-calling a model for prompts that already succeeded.
- **Structured logging** (vs. just `print`) matters because overnight jobs run when nobody's watching. A log line with a timestamp and an outcome is what someone reads at 8am to answer "did it work, and if not, where"; scrollback from `print` isn't searchable or saved anywhere.
- **The circuit breaker** is the piece most batch scripts skip -- and the one Priya explicitly asked for. Without it, a *systemic* failure (bad key, provider outage) doesn't just fail that one prompt -- it fails every remaining prompt in the queue, one at a time, burning through retries and rate-limit budget for nothing. Counting a *streak* (not a running total) is what distinguishes "one flaky prompt" from "everything is broken" -- a few scattered failures in 200 calls is normal and shouldn't trip anything.

---
## 🎫 Ticket #4 -- "Product wants this running unattended, overnight" (16 min) 🟢 core

New request in Slack from the product team: they want to run a batch of customer feedback prompts through the model overnight and have the results waiting in the morning. Priya's conditions:

> "If it crashes at item 40 of 200, I don't want to lose the first 39 -- checkpoint progress. I want to be able to tell you exactly which ones failed and why, without re-reading the whole log by eye -- log everything. And picture the OTHER overnight failure: item 12 starts throwing `AuthenticationError` because someone rotated the key wrong, and the job burns through the rest of the 200-item queue failing every single one, one at a time, all night. I don't want 188 identical failures in the log by 6am -- give up loudly after a few in a row and leave the rest of the queue untouched."

That's checkpointing + structured logging + a circuit breaker, all in one function, against real Groq calls.

**✏️ Build: `process_batch(client, prompts, checkpoint_every=3, max_consecutive_failures=3)`**

1. Iterate `prompts`, calling `client.chat(messages=[{"role": "user", "content": prompt}], max_tokens=30)` for each one.
2. Record one result dict per prompt **attempted**: on success, `{"prompt": prompt, "success": True, "reply": ..., "tokens": ...}`; on failure, `{"prompt": prompt, "success": False, "error": str(e)}`.
3. Every `checkpoint_every` items, print `[checkpoint] i/n done` -- so progress is visible even if the run dies mid-way.
4. Log every attempt with the `logger` set up below: `logger.info(...)` on success, `logger.error(...)` on failure.
5. Track **consecutive** failures -- a success resets the streak to 0. If the streak reaches `max_consecutive_failures`, **stop immediately**: do not call the API for the remaining prompts.
6. Return `{"results": [...], "stopped_early": True/False}`.

In [12]:
# ✏️ Ticket #4 -- your code here

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("llm_client")

def process_batch(client, prompts, checkpoint_every=3, max_consecutive_failures=3): # bouns " how to create batch processing with llm ?"
    results = []
    consecutive_failures = 0
    total = len(prompts)

    for i, prompt in enumerate(prompts):
        try:
            response = client.chat(
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=30
            )
            result = {
                "prompt": prompt,
                "success": True,
                "reply": response.choices[0].message.content,
                "tokens": response.usage.total_tokens
            }
            results.append(result)
            logger.info( f"SUCCESS prompt {i + 1}/{total}")
            consecutive_failures = 0
        except Exception as e:
            result = {"prompt": prompt, "success": False,"error": str(e) }
            results.append(result)

            logger.error( f"FAILED prompt {i + 1}/{total}: {e}")
            consecutive_failures += 1

            if consecutive_failures >= max_consecutive_failures:
                return { "results": results, "stopped_early": True }

        if (i + 1) % checkpoint_every == 0:
            print( f"[checkpoint] {i + 1}/{total} done")
    return {"results": results, "stopped_early": False
    }
    raise NotImplementedError

# Real prompts, standing in for the overnight customer-feedback queue product asked for.
batch_client = LLMClient(api_key=GROQ_API_KEY, base_delay=0.5)

prompts = [
    "Summarize this customer feedback in one sentence: 'The app crashed twice during checkout, very frustrating.'",
    "Classify the sentiment (positive/neutral/negative) of: 'Support replied in five minutes, fixed my issue instantly.'",
    "Summarize this customer feedback in one sentence: 'Pricing page is confusing, not sure which plan I need.'",
    "Classify the sentiment (positive/neutral/negative) of: 'Love the new dashboard, so much faster now.'",
    "Summarize this customer feedback in one sentence: 'Been waiting three days for a reply to my ticket.'",
]

# --- Acceptance test: do not edit below this line ---

# (a) Healthy run -- every prompt should be processed, breaker never trips.
healthy_out = process_batch(batch_client, prompts, checkpoint_every=3, max_consecutive_failures=3)
success_count = sum(r["success"] for r in healthy_out["results"])
total_tokens = sum(r.get("tokens", 0) for r in healthy_out["results"] if r["success"])
print(f"\nBatch complete: {success_count}/{len(prompts)} succeeded, {total_tokens} total tokens used "
      f"(real, billed against your free-tier quota)")

assert healthy_out["stopped_early"] is False, "a healthy run should not trip the breaker"
assert len(healthy_out["results"]) == len(prompts), "a healthy run should process every prompt"
print("✅ Healthy run processed every prompt -- checkpoint + logging output should be visible above.")

# (b) Broken client (bad key) -- every call fails fast (Ticket #3's behavior), so the streak climbs fast.
broken_client = LLMClient(api_key="invalid-key-definitely-wrong-12345", base_delay=0.3)
long_queue = [f"prompt {i}" for i in range(10)]
broken_out = process_batch(broken_client, long_queue, checkpoint_every=3, max_consecutive_failures=3)

assert broken_out["stopped_early"] is True, "breaker should have tripped on a broken client"
assert len(broken_out["results"]) == 3, (
    f"expected exactly 3 attempted results before stopping (max_consecutive_failures=3), "
    f"got {len(broken_out['results'])}"
)
assert all(r["success"] is False for r in broken_out["results"]), "all attempted results should be failures"
print(f"✅ PASSED -- breaker stopped after {len(broken_out['results'])} consecutive failures, "
      f"leaving {len(long_queue) - len(broken_out['results'])} prompts untouched instead of burning through all 10.")

[checkpoint] 3/5 done


ERROR:llm_client:FAILED prompt 1/10: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}



Batch complete: 5/5 succeeded, 620 total tokens used (real, billed against your free-tier quota)
✅ Healthy run processed every prompt -- checkpoint + logging output should be visible above.


ERROR:llm_client:FAILED prompt 2/10: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
ERROR:llm_client:FAILED prompt 3/10: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


✅ PASSED -- breaker stopped after 3 consecutive failures, leaving 7 prompts untouched instead of burning through all 10.


---
## 👀 Sneak peek -- "Wait, don't we have to build this every time?" (5 min, optional) 🔵 stretch

Priya swings by one more time, sees your `LLMClient` working, and grins. "Nice. Now imagine doing that for the *next* five projects -- a different retry wrapper, a different batch loop, a different prompt-formatting mess, every time. That's basically why the team's been evaluating **LangChain**."

**LangChain** is a library built specifically for this: talking to LLM providers, formatting prompts, and chaining steps together, without hand-rolling the plumbing every time. You are not replacing anything you just built -- you now understand *what's happening underneath* a library like this, which is exactly why Priya wanted you to build it by hand first. "I don't want engineers using a tool they can't debug when it breaks," she says.

We're not going deep today -- that starts next session. But here's the same idea from Ticket #1 and #2 (call a model, get a response), expressed the LangChain way, so the shape looks familiar when you see it for real.

In [13]:
%pip install -q langchain-openai langchain-core

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Same Groq endpoint, same model -- LangChain's ChatOpenAI wrapper works with
# any OpenAI-compatible API, same as the raw `openai` SDK did all session.
lc_model = ChatOpenAI(
    model=MODEL,
    api_key=GROQ_API_KEY,
    base_url=BASE_URL,
    max_retries=3,       # <- roughly your Ticket #2 retry loop, already built in
    temperature=0.3,
)

# LangChain's prompt templates replace the string-building you did by hand in Ticket #4
sentiment_prompt = ChatPromptTemplate.from_template(
    "Classify the sentiment (positive/neutral/negative) of this customer feedback: {feedback}"
)

# The "|" pipes the formatted prompt straight into the model call -- this is
# a LangChain "chain", the building block the rest of the course leans on.
chain = sentiment_prompt | lc_model

result = chain.invoke({"feedback": "Support replied in five minutes, fixed my issue instantly."})
print("Reply:", result.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 24.4 MB/s eta 0:00:00
Reply: positive


**What just happened, mapped to what you already built:**

| What you hand-built today | LangChain equivalent |
|---|---|
| `OpenAI(api_key=..., base_url=...)` | `ChatOpenAI(api_key=..., base_url=...)` |
| Your `chat()` retry loop (Ticket #2) | `max_retries=` on the model, handled internally |
| Manually f-string-ing prompts in `process_batch` (Ticket #4) | `ChatPromptTemplate` -- fill-in-the-blank prompts |
| Calling `.chat(...)` yourself | `chain.invoke(...)`, where `chain = prompt | model` |

Notice what's *missing* from the table: the fail-fast-vs-retry distinction from Ticket #3, the cost cap from Ticket #2, and the circuit breaker from Ticket #4. LangChain gives you hooks for some of these, but it doesn't make the decisions for you -- you still need to know *why* a 401 shouldn't retry and a 429 should, which is exactly what today was for.

**Next session** picks this up properly: prompt templates, chains, and where LangChain's abstractions genuinely save you time versus where they hide something you need to see. For now, just notice the shape looks familiar -- because it should.

---
## Summary

| Ticket | What you delivered |
|---|---|
| #1 -- Understand the payload | Made real API calls and built `summarize_usage()` to audit token spend across them, with asserts proving it |
| #2 -- One client class | Built `LLMClient.chat()` yourself: real exponential-backoff retries, fail-fast on 401/400, AND a hard `max_tokens_cap` that rejects oversized requests before they hit the network |
| #3 -- Prove it | Threw three REAL failures (401, 400, 429) at the exact `chat()` you wrote and proved it survives each correctly, then built `total_wait_time()` to quantify backoff cost |
| #4 -- Overnight batch job | Built `process_batch()` yourself: checkpointing, structured logging, AND a circuit breaker that stops a run after consecutive failures instead of burning through the whole queue |
| Sneak peek | Saw the same call/retry/prompt pattern expressed with LangChain -- the abstraction, now that you know what it's hiding |
| Real-world messiness | Saw firsthand that real rate limits aren't perfectly deterministic -- and that your code handles that correctly anyway |

**A note on verification:** every cell in this notebook was written against Groq's documented, OpenAI-compatible API and the `openai` Python SDK's real exception classes. If your results differ slightly from the descriptions above (exact 429 timing, specific error message text), that's expected -- providers' exact limits and messages can change over time. Focus on whether your code's PATTERN of behavior (retry vs. fail-fast) is correct.

**Next:** the assignment ("Production Client Library") asks you to extend this exact client with a new resilience feature and stress-test it against real triggered failures. Picture Priya reviewing the PR -- what would she ask you to prove?

*Module 4 - Session 1 -- Generative & Agentic AI Systems*

---
## 🔑 Answer Key (spoilers -- try the exercise first)

### Exercise 1.1 -- `summarize_usage()`

**The idea:** every response object already carries a `.usage` field (you printed one in Ticket #1's Checkpoint) with `.prompt_tokens`, `.completion_tokens`, and `.total_tokens` on it. The whole exercise is just "loop over the responses you're given, add up each field, and compare the total to the budget" -- no new API calls needed, which is why the function doesn't take `client` or `model` as arguments at all.

A common trap: computing `over_budget` from `total_prompt_tokens + total_completion_tokens` by hand instead of using `total_tokens` (or trusting the provider's own total) -- they should always agree, but if they ever don't, that mismatch itself would be worth surfacing, not silently recomputing over it.

### Piece 2 -- `LLMClient.chat()`

**The idea:** two checks happen before you ever loop -- well, one check (the cap) happens before the loop, and the retry loop itself has exactly two `except` branches, ordered from most-specific to least. The trap most people fall into: putting the `RateLimitError`/`APIStatusError` branch first. Since `AuthenticationError` and `BadRequestError` are themselves subclasses of `APIStatusError`, a broader branch listed first would silently swallow them and retry a request that can never succeed -- burning 3 retries and ~7s of backoff on a bad API key instead of failing in <1s.

### Exercise 3.1 -- `total_wait_time()`

**The idea:** a one-line generator sum over `call_log`, filtering to `"retry"` entries and re-deriving `wait` from the stored `attempt` (remembering it's 1-indexed in the log but the formula is 0-indexed).

### Ticket #4 -- `process_batch()`

**The idea:** a single `consecutive_failures` counter, incremented on failure and reset to `0` on success. The breaker check happens *inside* the loop, right after recording a failure -- checking it, and returning early, before moving on to the next prompt. Note the `return` (not `break`) inside the exception handler: it needs to skip the checkpoint-print for that same iteration too, which is why the early-exit happens before the `(i + 1) % checkpoint_every` check.

A common trap: putting `consecutive_failures = 0` in the `try` block *before* the API call instead of only on confirmed success -- that would reset the streak even when the call itself raises, defeating the whole breaker.